<a href="https://colab.research.google.com/github/cyanm002-maker/Deep_learning_project/blob/main/nvidia_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np



In [ ]:
data = pd.read_csv("/content/drive/MyDrive/Nvidia_hackathon/data.csv")

In [ ]:
data.head()

In [ ]:
data.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Dataset Overview & Missing Values
print("--- Dataset Info ---")
print(data.info())

print("\n--- Summary Statistics ---")
display(data.describe())

print("\n--- Missing Values Check ---")
print(data.isnull().sum())

# 2. Visualize Target Variable Distributions
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(data['total_UPDRS'], kde=True, color='blue')
plt.title('Distribution of total_UPDRS')

plt.subplot(1, 2, 2)
sns.histplot(data['motor_UPDRS'], kde=True, color='orange')
plt.title('Distribution of motor_UPDRS')

plt.tight_layout()
plt.show()

# 3. Correlation Heatmap (Focusing on features and targets)
plt.figure(figsize=(14, 10))
# Selecting a subset or all numeric columns for correlation
numeric_data = data.select_dtypes(include=['float64', 'int64'])
correlation_matrix = numeric_data.corr()

# Plotting heatmap for targets and top features
sns.heatmap(
    correlation_matrix[['total_UPDRS', 'motor_UPDRS']],
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    linewidths=0.5,
)
plt.title('Feature Correlation with UPDRS Scores')
plt.show()

# feature engineering



In [ ]:
import numpy as np
import pandas as pd

# seprating the features
drop_cols = ['subject#', 'motor_UPDRS', 'total_UPDRS']
X = data.drop(columns = drop_cols,errors = 'ignore').copy()
y = data['total_UPDRS']

if 'HNR' in X.columns and 'NHR' in X.columns:
  X['HNR_NHR_ratio'] = X['HNR'] / (X['NHR'] + 1e-5)

# Combined Shimmer and Jitter aggregate metrics
jitter_cols = [c for c in X.columns if 'Jitter' in c]
shimmer_cols = [c for c in X.columns if 'Shimmer' in c]

if jitter_cols:
  X['jitter_mean'] = X[jitter_cols].mean(axis=1)
if shimmer_cols:
  X['shimmer_mean'] = X[shimmer_cols].mean(axis=1)

print('Engineered features added successfully!')
print(f'New feature set shape: {X.shape}')
print('Features:', list(X.columns))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

groups = data['subject#']

gas = GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
train_idx,test_idx = next(gas.split(X,y,groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# converting data into pytorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# Create DataLoaders for batching
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

class UPDRSRegressor(nn.Module):

  def __init__(self, input_dim):
    super(UPDRSRegressor, self).__init__()
    self.network = nn.Sequential(
        nn.Linear(input_dim, 64),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(32, 1),  # Output layer for continuous regression
    )

  def forward(self, x):
    return self.network(x)

# Setup device (utilizes Colab's T4 GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UPDRSRegressor(input_dim=X_train_scaled.shape[1]).to(device)

# Loss and Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 6. Training Loop with Early Stopping / Convergence tracking
epochs = 100
best_loss = float('inf')
patience, patience_counter = 15, 0

for epoch in range(epochs):
  model.train()
  epoch_loss = 0
  for batch_x, batch_y in train_loader:
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    # Forward pass
    predictions = model(batch_x)
    loss = criterion(predictions, batch_y)

    # Backward pass & optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item() * batch_x.size(0)

  epoch_loss /= len(train_loader.dataset)

  if (epoch + 1) % 10 == 0:
    print(f'Epoch [{epoch + 1}/{epochs}], Train MSE Loss: {epoch_loss:.4f}')

# 7. Evaluate on Unseen Test Set
model.eval()
with torch.no_grad():
  X_test_tensor = X_test_tensor.to(device)
  y_test_tensor = y_test_tensor.to(device)
  test_preds = model(X_test_tensor)
  test_mse = criterion(test_preds, y_test_tensor).item()
  test_rmse = np.sqrt(test_mse)

print(f'\nFinal Test RMSE (PyTorch): {test_rmse:.4f}')



# RandomForest Classifier

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np


rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

rf_preds = rf_model.predict(X_test_scaled)


rf_mse = mean_squared_error(y_test, rf_preds)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_preds)

print(f"Random Forest Test MSE: {rf_mse:.4f}")
print(f"Random Forest Test RMSE: {rf_rmse:.4f}")
print(f"Random Forest R² Score: {rf_r2:.4f}")

# XGBoost

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# 1. Initialize and train the XGBoost Regressor
# Using standard hyperparameters optimized for regression
xgb_model = xgb.XGBRegressor(
    n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1
)

xgb_model.fit(X_train_scaled, y_train)

# 2. Predict on the unseen test set
xgb_preds = xgb_model.predict(X_test_scaled)

# 3. Evaluate performance
xgb_mse = mean_squared_error(y_test, xgb_preds)
xgb_rmse = np.sqrt(xgb_mse)
xgb_r2 = r2_score(y_test, xgb_preds)

print(f"XGBoost Test MSE: {xgb_mse:.4f}")
print(f"XGBoost Test RMSE: {xgb_rmse:.4f}")
print(f"XGBoost R² Score: {xgb_r2:.4f}")

# gradient boost

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.svm import SVR
import numpy as np

# Dictionary of multiple models to evaluate
models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100, random_state=42
    ),
    'Support Vector Regressor': SVR(kernel='rbf', C=1.0, epsilon=0.1),
}

results = []

for name, model in models.items():
  # Train the model
  model.fit(X_train_scaled, y_train)

  # Predict on test set
  preds = model.predict(X_test_scaled)

  # Calculate metrics
  mse = mean_squared_error(y_test, preds)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, preds)

  results.append({'Model': name, 'MSE': mse, 'RMSE': rmse, 'R2': r2})
  print(f'--- {name} ---')
  print(f'MSE: {mse:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}\n')

# SVR Model test and train accracy

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.svm import SVR
import numpy as np

# 1. Train the SVR model (our best performer)
svr_model = SVR(kernel='rbf', C=1.0, epsilon=0.1)
svr_model.fit(X_train_scaled, y_train)

# 2. Make predictions on BOTH Train and Test sets
train_preds = svr_model.predict(X_train_scaled)
test_preds = svr_model.predict(X_test_scaled)

# 3. Calculate metrics for Training Set
train_mse = mean_squared_error(y_train, train_preds)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, train_preds)

# 4. Calculate metrics for Test Set
test_mse = mean_squared_error(y_test, test_preds)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, test_preds)

# 5. Print out the comparison
print('=== TRAINING SET PERFORMANCE ===')
print(f'Train RMSE: {train_rmse:.4f} | Train R²: {train_r2:.4f}\n')

print('=== UNSEEN TEST SET PERFORMANCE ===')
print(f'Test RMSE:  {test_rmse:.4f} | Test R²:  {test_r2:.4f}')

In [ ]:
import joblib

# 1. Save the best-performing SVR model and the fitted scaler
joblib.dump(svr_model, "svr_updrs_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print(
    "Model and scaler successfully saved to disk! You can now download them"
    " from the file explorer."
)

In [ ]:
# Install Gradio
!pip install gradio

import gradio as gr

# Define your prediction function (plug your model logic here)
def predict(text_input):
    # Example: run your model inference here
    return f"Model Output for: {text_input}"

# Create the UI and set share=True to get a free public link
demo = gr.Interface(
    fn=predict,
    inputs="textbox",
    outputs="textbox",
    title="My Hackathon Model Demo"
)

# share=True creates a public live link that judges can access from anywhere!
demo.launch(share=True)

In [ ]:
import gradio as gr
import joblib
import numpy as np
import pandas as pd

# 1. Load the saved model and scaler
model = joblib.load("svr_updrs_model.pkl")
scaler = joblib.load("scaler.pkl")

# 2. Define the feature names matching your training pipeline
feature_names = [
    "age",
    "sex",
    "test_time",
    "Jitter(%)",
    "Jitter(Abs)",
    "Jitter:RAP",
    "Jitter:PPQ5",
    "Jitter:DDP",
    "Shimmer",
    "Shimmer(dB)",
    "Shimmer:APQ3",
    "Shimmer:APQ5",
    "Shimmer:APQ11",
    "Shimmer:DDA",
    "NHR",
    "HNR",
    "RPDE",
    "DFA",
    "PPE",
    "HNR_NHR_ratio",
    "jitter_mean",
    "shimmer_mean",
]


# 3. Define the prediction function
def predict_updrs(*inputs):
  # Convert inputs to a DataFrame with proper column names
  input_df = pd.DataFrame([inputs], columns=feature_names)

  # Scale the input data using the saved scaler
  input_scaled = scaler.transform(input_df)

  # Predict using the SVR model
  prediction = model.predict(input_scaled)

  return float(prediction[0])


# 4. Create Gradio inputs (one number input for each of the 22 features)
inputs = [
    gr.Number(label=col, value=0.0) for col in feature_names
]  # You can set default/mean values if preferred

# 5. Launch the Gradio Interface
demo = gr.Interface(
    fn=predict_updrs,
    inputs=inputs,
    outputs=gr.Number(label="Predicted total_UPDRS Score"),
    title="Parkinson's Sound Biomarker UPDRS Predictor",
    description=(
        "Enter the voice feature metrics extracted from the audio recordings to"
        " predict the total_UPDRS score."
    ),
)

demo.launch(share=True)